In [126]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels
%pip install linearmodels
%pip install stargazer

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS
from stargazer.stargazer import Stargazer
import matplotlib.pyplot as plt
import pickle


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [137]:
################ Import data
df = pd.read_csv('model_ready.csv')
supply_df = pd.read_csv('supply_ready.csv')

In [138]:
# Check for variations in number of available products over time and across provinces

# 统计每个省份每年可用产品数
product_counts = df.groupby(['province_ids', 'year'])['product_ids'].nunique().reset_index(name='num_products')
print(product_counts)

# 检查每年不同省份的产品数分布
print("\n产品数在每年各省的分布：")
print(product_counts.groupby('year')['num_products'].describe())

# 检查每个省份不同年份的产品数分布
print("\n产品数在各省每年的分布：")
print(product_counts.groupby('province_ids')['num_products'].describe())

    province_ids  year  num_products
0            P01  2019           114
1            P01  2020           146
2            P01  2021           187
3            P01  2022           184
4            P01  2023           228
..           ...   ...           ...
150          P31  2019           107
151          P31  2020           134
152          P31  2021           174
153          P31  2022           172
154          P31  2023           212

[155 rows x 3 columns]

产品数在每年各省的分布：
      count        mean        std    min    25%    50%    75%    max
year                                                                 
2019   31.0  114.387097   6.591294   96.0  111.5  116.0  118.0  123.0
2020   31.0  150.677419  12.343384  117.0  144.0  155.0  160.0  163.0
2021   31.0  188.000000  13.271523  151.0  184.0  193.0  197.5  204.0
2022   31.0  180.193548  12.666016  143.0  175.5  185.0  188.5  193.0
2023   31.0  240.838710  22.737776  162.0  230.5  249.0  255.5  266.0

产品数在各省每年的分布：
              

In [139]:
# 将 supply_df 中的 road_fuel_IV 合并到 df
# 假设合并键为 province 和 year

df = df.merge(
    supply_df[['province', 'year', 'road_fuel_IV']],
    on=['province', 'year'],
    how='left'
)

# 计算 log(sjm), log(s0m), log(sj/g)
df['log_sjm'] = np.log(df['shares'])
df['log_s0m'] = np.log(1 - df.groupby('market_ids')['shares'].transform('sum'))
df['log_sj_g'] = np.log(df['shares'] / df.groupby(['market_ids', 'nesting_ids'])['shares'].transform('sum'))
df['log_charging_stock'] = np.log(df['charging_stations_stock'])

# Set range, log_charging_stock, log_charging_IV, and battery_capacity to 0 for non-EVs
df.loc[df['is_electric'] == 0, ['range', 'battery_capacity']] = 0

In [140]:
# Define list of IVs
iv_list = [
    'cost_shifter', 
    'product_set_size',
    'euclidean_range',
    'local_range',
    'euclidean_battery',
    'local_battery',
    'euclidean_power',
    'local_power',
    'road_fuel_IV',
]

iv_str = ' + '.join(iv_list)

In [141]:
################ 2SLS model using custom instruments

# First stage for charging station: regress log_charging_stock on instrument variables
X = sm.add_constant(df[['log_charging_IV']])
y = df['log_charging_stock']
first_stage = sm.OLS(y, X).fit()
df['log_charging_stock_hat'] = first_stage.predict(X)

# Define the formula for the IV2SLS model
formula = f'''
(log_sjm - log_s0m) ~ 0 + is_electric*log_charging_stock_hat + range + power + battery_capacity
    + [net_prices + log_sj_g ~ {iv_str}]
'''
# CAN HAVE HORSEPOWER = POWER / MASS #

iv_model = IV2SLS.from_formula(formula, data=df).fit(cov_type="clustered", clusters=df['market_ids'])

# 
main_vars = ['Intercept', 'net_prices', 'power', 'range', 'battery_capacity', 'log_charging_IV', 'log_sj_g']

print(iv_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                log_sjm   R-squared:                      0.9811
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9811
No. Observations:               33545   F-statistic:                 7.262e+04
Date:                Wed, Jul 30 2025   P-value (F-stat)                0.0000
Time:                        02:48:42   Distribution:                  chi2(8)
Cov. Estimator:             clustered                                         
                                                                              
                                         Parameter Estimates                                          
                                    Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------------------------
is_electric                           -15.821     0.6575   

In [142]:
# Check first stage results
print(iv_model.first_stage.summary)

                First Stage Estimation Results                
                                       net_prices     log_sj_g
--------------------------------------------------------------
R-squared                                  0.8390       0.8950
Partial R-squared                          0.1002       0.0644
Shea's R-squared                           0.1015       0.0653
Partial F-statistic                        701.39       339.48
P-value (Partial F-stat)                   0.0000       0.0000
Partial F-stat Distn                      chi2(9)      chi2(9)
==================================== ============ ============
is_electric                               -15.809      -3.6543
                                        (-9.6543)    (-10.279)
log_charging_stock_hat                    -0.7219      -0.3369
                                        (-7.2975)    (-15.817)
range                                      0.0082       0.0028
                                         (6.4855)     (

In [143]:
################ Supply side model

# Create a time trend variable
supply_df['time_trend'] = supply_df['year'].astype(int) - supply_df['year'].astype(int).min() + 1

# Add time_trend
formula = 'log(charging_stations_stock) ~ [log(EV_stock) ~ road_fuel_IV + num_models_in_market + sales_weighted_avg_range] + sub_fix + sub_ope + C(province) + time_trend'

supply_model = IV2SLS.from_formula(formula, data=supply_df).fit()
print(supply_model.summary)
print(supply_model.first_stage.summary)

                               IV-2SLS Estimation Summary                               
Dep. Variable:     log(charging_stations_stock)   R-squared:                      0.9734
Estimator:                              IV-2SLS   Adj. R-squared:                 0.9659
No. Observations:                           155   F-statistic:                 9.753e+05
Date:                          Wed, Jul 30 2025   P-value (F-stat)                0.0000
Time:                                  02:49:31   Distribution:                 chi2(35)
Cov. Estimator:                          robust                                         
                                                                                        
                                   Parameter Estimates                                   
                       Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------
sub_fix           

In [144]:
# Export model summary as LaTeX
# Stargazer does not support different covariate orders for each model,
# so the best practice is to include all you want to show and accept blanks for the other.
main_vars = ['sub_fix', 'sub_ope', 'time_trend', 'log(EV_stock)']

stargazer = Stargazer([supply_model])
stargazer.covariate_order(main_vars)

with open('supply_model_stargazer.tex', 'w', encoding='utf-8') as f:
    f.write(stargazer.render_latex())


In [145]:
# 保存需求侧模型结果
with open('demand_model_results.pkl', 'wb') as f:
    pickle.dump(iv_model, f)

# 保存供给侧模型结果
with open('charging_station_model_results.pkl', 'wb') as f:
    pickle.dump(supply_model, f)

In [146]:
# Export model data for counterfactual analysis
df.to_csv('demand_counterfactual.csv', index=False)
supply_df.to_csv('supply_counterfactual.csv', index=False)